In [ ]:
import csv
import time
import random
import re
import threading
from queue import Queue
from urllib.parse import quote
import pandas as pd
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import TimeoutException

# ============================================================
#  并行配置
# ============================================================
NUM_PRODUCERS = 2
NUM_CONSUMERS = 5
TARGET_KEYWORDS = ["数据分析", "数据科学", "算法", "风控"]
OUTPUT_FILE = "tencent_campus.csv"

# ============================================================
# 工具函数
# ============================================================

def get_test_driver():
    options = webdriver.ChromeOptions()
    options.add_argument("--disable-blink-features=AutomationControlled")
    options.add_experimental_option("excludeSwitches", ["enable-automation"])
    options.add_experimental_option('useAutomationExtension', False)
    options.add_argument('--no-sandbox')
    options.add_argument('--disable-gpu')
    options.add_argument('--disable-dev-shm-usage')
    options.add_argument("--user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/148.0.0.0 Safari/537.36")
    options.page_load_strategy = 'eager'
    
    driver_path = "C:/Program Files/Google/Chrome/Application/chromedriver.exe"
    service = Service(executable_path=driver_path)
    driver = webdriver.Chrome(service=service, options=options)
    driver.set_page_load_timeout(30)
    
    driver.execute_cdp_cmd("Page.addScriptToEvaluateOnNewDocument", {
        "source": """
            Object.defineProperty(navigator, 'webdriver', { get: () => undefined });
            window.chrome = { runtime: {} };
            Object.defineProperty(navigator, 'plugins', { get: () => [1, 2, 3, 4, 5] });
            Object.defineProperty(navigator, 'languages', { get: () => ['zh-CN', 'zh', 'en'] });
        """
    })
    
    return driver


FIELD_NAMES = ["职位名称", "工作城市", "职能类别", "所属部门", "工作经验",
               "发布时间", "职位简述", "职位要求", "postId", "详情页URL", "搜索关键词"]

BASE_LIST_URL = "https://careers.tencent.com/search.html?index={page}&keyword={keyword}"


def scrape_detail_from_tab(driver, list_tab):
    """在详情页标签中提取数据，完成后关标签回到列表页"""
    detail_url = driver.current_url
    
    post_id = ""
    m = re.search(r'postId=(\d+)', detail_url)
    if m:
        post_id = m.group(1)
    
    try:
        WebDriverWait(driver, 10).until(
            EC.presence_of_element_located((By.CSS_SELECTOR, '.duty-text, .requirement'))
        )
        time.sleep(0.5)
        
        # ---- 职位名称 ----
        job_title = "未知"
        try: job_title = driver.find_element(By.CSS_SELECTOR, '.job-recruit-title').text.strip()
        except: pass
        
        # ---- 工作城市 ----
        city = "未知"
        try: city = driver.find_element(By.CSS_SELECTOR, '.job-recruit-location').text.strip()
        except: pass
        
        # ---- 基本信息 ----
        dept = "未知"; func_cat = "未知"; exp = "未知"; date = "未知"
        try:
            tips_container = driver.find_element(By.CSS_SELECTOR, '.work-wrapper .recruit-tips')
            children = tips_container.find_elements(By.XPATH, './*')
            tips = []
            for c in children:
                cls = c.get_attribute('class') or ''
                if 'vertical-split-line' in cls:
                    continue
                if 'bg-tip' in cls:
                    try:
                        txt = c.find_element(By.CSS_SELECTOR, '.reference').text.strip()
                    except:
                        txt = c.text.strip()
                else:
                    txt = c.text.strip()
                if txt:
                    tips.append(txt)
            
            if len(tips) >= 1: dept = tips[0]
            if len(tips) >= 2: func_cat = tips[1]
            for t in tips[2:]:
                if '经验' in t: exp = t
                elif '更新于' in t: date = t.replace('更新于', '').strip()
        except: pass
        
        # ---- 岗位职责 & 岗位要求 ----
        jd_text = "未知"; job_requirements = "未知"
        try:
            duty = driver.find_element(By.CSS_SELECTOR, '.duty.work-module .duty-text')
            items = duty.find_elements(By.CSS_SELECTOR, 'li.explain-item')
            jd_text = '\n'.join([it.text.strip() for it in items if it.text.strip()]) if items else duty.text.strip()
        except: pass
        
        try:
            req = driver.find_element(By.CSS_SELECTOR, '.requirement.work-module .duty-text')
            items = req.find_elements(By.CSS_SELECTOR, 'li.explain-item')
            job_requirements = '\n'.join([it.text.strip() for it in items if it.text.strip()]) if items else req.text.strip()
        except: pass
        
        return {
            "职位名称": job_title, "工作城市": city,
            "职能类别": func_cat, "所属部门": dept,
            "工作经验": exp, "发布时间": date,
            "职位简述": jd_text, "职位要求": job_requirements,
            "post_id": post_id, "detail_url": detail_url
        }
        
    except Exception as e:
        return {"职位名称": "解析失败", "工作城市": "未知", "职能类别": "未知",
                "所属部门": "未知", "工作经验": "未知", "发布时间": "未知",
                "职位简述": "解析失败", "职位要求": "解析失败",
                "post_id": post_id, "detail_url": detail_url}
    finally:
        driver.close()
        driver.switch_to.window(list_tab)


def split_keywords(keywords, num_producers):
    chunks = []; start = 0
    base = len(keywords) // num_producers
    remainder = len(keywords) % num_producers
    for i in range(num_producers):
        size = base + (1 if i < remainder else 0)
        chunks.append(keywords[start:start + size])
        start += size
    return [c for c in chunks if c]


# ============================================================
# 生产者：列表页翻页 → 点卡片 → 新标签页抓详情 → 关标签页回来
# ============================================================

WAIT_PAGE = 20

def producer_worker(keywords, task_queue, stats, producer_id):
    driver = get_test_driver()
    total = 0
    try:
        for kw in keywords:
            tprint(f"[P{producer_id}] 开始收集: {kw}")
            kw_start = total
            page = 1
            
            while True:
                url = BASE_LIST_URL.format(keyword=quote(kw), page=page)
                driver.get(url)
                
                try:
                    WebDriverWait(driver, WAIT_PAGE).until(
                        EC.presence_of_element_located((By.CSS_SELECTOR, '.recruit-list'))
                    )
                except TimeoutException:
                    tprint(f"[P{producer_id}] {kw} page={page} 超时/无数据")
                    break
                
                cards = driver.find_elements(By.CSS_SELECTOR, '.recruit-list')
                if not cards: break
                
                list_tab = driver.current_window_handle
                page_count = 0
                
                for card in cards:
                    try:
                        link = card.find_element(By.CSS_SELECTOR, '.recruit-list-link')
                        driver.execute_script("arguments[0].click();", link)
                        time.sleep(1.5)
                        
                        handles = driver.window_handles
                        if len(handles) > 1:
                            new_tab = [h for h in handles if h != list_tab][0]
                            driver.switch_to.window(new_tab)
                            
                            job_data = scrape_detail_from_tab(driver, list_tab)
                            task_queue.put({**job_data, "keyword": kw})
                            page_count += 1; total += 1
                            
                            time.sleep(random.uniform(1, 2))
                    except: continue
                
                with print_lock: stats['enqueued'] += page_count
                
                if page_count < 10: break
                page += 1
                time.sleep(random.uniform(1, 3))
            
            tprint(f"[P{producer_id}] {kw} 完成，入队 {total - kw_start} 条")
                
    except Exception as e:
        import traceback; tprint(f"[P{producer_id}] 异常: {e}"); traceback.print_exc()
    finally:
        driver.quit()
    
    tprint(f"[P{producer_id}] 完成，共入队 {total} 条")


# ============================================================
# 消费者：写入 CSV
# ============================================================

def consumer_worker(task_queue, csv_writer, file_lock, stats, consumer_id):
    processed = 0
    try:
        while True:
            item = task_queue.get()
            if item is None:
                task_queue.task_done()
                break
            
            try:
                row = {
                    "职位名称": item.get("职位名称", "未知"),
                    "工作城市": item.get("工作城市", "未知"),
                    "职能类别": item.get("职能类别", "未知"),
                    "所属部门": item.get("所属部门", "未知"),
                    "工作经验": item.get("工作经验", "未知"),
                    "发布时间": item.get("发布时间", "未知"),
                    "职位简述": item.get("职位简述", "未知"),
                    "职位要求": item.get("职位要求", "未知"),
                    "postId": item.get("post_id", ""),
                    "详情页URL": item.get("detail_url", ""),
                    "搜索关键词": item.get("keyword", "")
                }
                
                with file_lock:
                    csv_writer.writerow(row)
                    stats['written'] += 1
                processed += 1
            except: pass
            finally:
                task_queue.task_done()
    except Exception as e:
        import traceback; tprint(f"[C{consumer_id}] 异常: {e}"); traceback.print_exc()
    
    tprint(f"[C{consumer_id}] 退出，处理 {processed} 条")


# ============================================================
# 统计报告 & 打印
# ============================================================

print_lock = threading.Lock()

def tprint(msg):
    with print_lock:
        print(msg)

def stats_reporter(stats, stop_event):
    while not stop_event.is_set():
        stop_event.wait(10)
        if stop_event.is_set(): break
        with print_lock:
            print(f"[STATS] 入队 {stats['enqueued']} | 已写 {stats['written']} | 队列 ~{stats['enqueued'] - stats['written']}")


# ============================================================
# 主函数
# ============================================================

def run_parallel_crawl():
    producer_assignments = split_keywords(TARGET_KEYWORDS, NUM_PRODUCERS)
    
    task_queue = Queue()
    stats = {'enqueued': 0, 'written': 0}
    stop_event = threading.Event()
    
    csv_file = open(OUTPUT_FILE, 'w', newline='', encoding='utf-8-sig')
    csv_writer = csv.DictWriter(csv_file, fieldnames=FIELD_NAMES)
    csv_writer.writeheader(); csv_file.flush()
    file_lock = threading.Lock()
    
    reporter = threading.Thread(target=stats_reporter, args=(stats, stop_event), daemon=True)
    reporter.start()
    
    producers = []
    for pid, kws in enumerate(producer_assignments, 1):
        t = threading.Thread(target=producer_worker, args=(kws, task_queue, stats, pid), daemon=True)
        t.start(); producers.append(t)
    
    tprint(f"生产者已启动 ({len(producers)} 线程)")
    time.sleep(5)
    
    consumers = []
    for cid in range(1, NUM_CONSUMERS + 1):
        t = threading.Thread(target=consumer_worker, args=(task_queue, csv_writer, file_lock, stats, cid), daemon=True)
        t.start(); consumers.append(t)
    
    tprint(f"消费者已启动 ({NUM_CONSUMERS} 线程)")
    tprint("=" * 50)
    
    for t in producers: t.join()
    
    tprint("=" * 50)
    tprint("生产者全部完成，等待消费者处理剩余任务...")
    
    for _ in range(NUM_CONSUMERS): task_queue.put(None)
    for t in consumers: t.join()
    
    stop_event.set()
    csv_file.close()
    
    tprint(f"全部完成，入队 {stats['enqueued']} 条，写入 {stats['written']} 条")
    return OUTPUT_FILE


if __name__ == "__main__":
    result_file = run_parallel_crawl()
    try:
        df = pd.read_csv(result_file, encoding='utf-8-sig')
        if len(df) > 0:
            df.drop_duplicates(subset=['postId'], inplace=True)
            df.to_csv(result_file, index=False, encoding='utf-8-sig')
            print(f"\n去重后共计 {len(df)} 条 -> {result_file}")
        else:
            print("\n未能获取到任何数据")
    except Exception as e:
        print(f"\n读取结果文件失败: {e}")